In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns

import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

from studio_preprocessing_tools import *

In [2]:
Nramp_dataset = pd.read_csv("../../pg2/DMS_assays/raw_data/MNTH_DEIRA_Berry_2025/merged_Mn_Mg_scores_all.csv", index_col=0)
# l1_volcano_hits = pd.read_csv("L1_all_significant_hits.csv")
# l2v2_volcano_hits = pd.read_csv("L2v2_all_significant_hits.csv")
# l2_volcano_hits = pd.read_csv("L2_all_significant_hits.csv")

l1_volcano_hits = pd.read_csv("../L1_volcano_hits.csv")
l2v2_volcano_hits = pd.read_csv("../L2_volcano_hits.csv")

In [3]:
v_hits = np.unique(pd.concat([l1_volcano_hits, l2v2_volcano_hits], axis=0).variant)
single_v_hits = [i for i in v_hits if ',' not in i and 'ins' not in i and 'del' not in i and 'del' not in i and 'no_start' not in i and '_v' not in i]
double_v_hits = [i for i in v_hits if i.count(',')==1 and 'ins' not in i and 'del' not in i and 'del' not in i and 'no_start' not in i and '_v' not in i]

Nramp_dataset = Nramp_dataset.loc[[i for i in Nramp_dataset.index if 'no_start' not in i and 'ins' not in i and 'del' not in i and '_v' not in i]]

Nramp_dataset_M230X = Nramp_dataset.loc[[i for i in Nramp_dataset.index if 'M230' in i]]
Nramp_dataset_noM230X = Nramp_dataset.loc[[i for i in Nramp_dataset.index if 'M230' not in i]]

print(len(Nramp_dataset_M230X), len(Nramp_dataset_noM230X))

7436 20099


In [4]:
def is_specificity_hit(var,row, mg_thresh=2.0):
    if 'M230' in var:
        if row.Mg_import_score > mg_thresh:
            return True
        else:
            return False
    else:
        if var in v_hits:
            return True
        else:
            return False

Nramp_dataset = assign_specificity_categories(Nramp_dataset, 
                              lambda var,row:row.Mn_import_score<0.2 and row.Mg_import_score<2,
                              is_specificity_hit,
                               exclude=lambda var,row:row.Mg_import_score>2)

In [5]:
Nramp_dataset = assign_mutant_column(Nramp_dataset)
Nramp_dataset['mutated_sequence'] = Nramp_dataset['aa_seq'].copy()
Nramp_dataset['DMS_score'] = Nramp_dataset['Mn_import_score']

check_dataset(Nramp_dataset)

Nramp_dataset.to_csv("../../processed_data/MNTH_DEIRA_Berry_2025.csv")

Checking dataset table:
✅ DMS_score in table
✅ mutated_sequence in table
✅ mutant in table
✅ Contains WT sequence (M1M)
✅ all mutants pass checks and match sequence

✅ All checks passed!


In [6]:
np.savetxt('../../processed_data/Nramp_ESCOTT.mut', Nramp_dataset.mutant.values, fmt="%s")
np.savetxt('../../processed_data/Nramp_GEMME.mut', [i.replace(':',',') for i in Nramp_dataset.mutant.values], fmt="%s")